# MLOps Assignment 2: LLM API Evaluation

This experiment compares three prompts for an academic question-answering assistant using Gemini. It measures response time, token usage, cost and JSON validity, then records the results in MLflow.

LLMOps adds prompt versions, API models, context and response checks to the usual MLOps concerns of reproducibility, deployment and monitoring.

**Setup:** use Python 3.10–3.13 and place a `.env` file beside this notebook containing `GEMINI_API_KEY=your_key`. Copy the supplied `ENV.example` to make this file. In Colab, upload `.env` into `/content` before running. Keep the key out of the notebook and submission.

For Task 1, I use the basic prompt (v1) for five questions. Task 2 uses those responses. Task 3 reuses v1 and evaluates v2 and v3 on the same questions, giving 15 API requests in total.

## Setup: dependencies
Install the required package versions in the current Python kernel. Restart the kernel if installation requests it, then run the notebook again.

In [1]:
import sys, subprocess, importlib.metadata
assert (3,10) <= sys.version_info[:2] < (3,14), "Use Python 3.10–3.13"
REQUIREMENTS = 'mlflow==3.5.1\npandas==2.3.3\nSQLAlchemy==2.0.44\npython-dotenv==1.2.4\ncertifi==2026.10.23\nipykernel==6.30.1\n'
missing = []
for spec in REQUIREMENTS.splitlines():
    if not spec.strip(): continue
    package, version = spec.split("==")
    try:
        if importlib.metadata.version(package) != version: missing.append(spec)
    except importlib.metadata.PackageNotFoundError:
        missing.append(spec)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Python:", sys.version.split()[0], "| Dependencies ready")

Python: 3.13.5 | Dependencies ready


## Setup: environment variables
Read the key from `.env` using python-dotenv. Change `ENV_FILE` if the file is stored elsewhere.

In [2]:
from pathlib import Path
import os, json, math, time, uuid, difflib, hashlib, csv
from datetime import datetime, timezone
from dotenv import load_dotenv
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
ENV_FILE = Path(os.getenv("EDUASSIST_ENV_FILE", str(ROOT / ".env"))).expanduser()
if not ENV_FILE.is_file():
    raise FileNotFoundError("Supply your existing .env before Run All. In Colab upload it using the Files panel; locally place it beside the notebook.")
load_dotenv(ENV_FILE, override=True, interpolate=False)
API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
if not API_KEY or API_KEY in {"your_actual_key", "replace_with_your_gemini_api_key"}:
    raise ValueError("Set GEMINI_API_KEY in your existing .env. Do not put the key in this notebook.")
REQUEST_TIMEOUT = float(os.getenv("GEMINI_REQUEST_TIMEOUT_SECONDS", "120"))
if not math.isfinite(REQUEST_TIMEOUT) or REQUEST_TIMEOUT <= 0:
    raise ValueError("Request timeout must be finite and positive")
INTERVAL = float(os.getenv("GEMINI_REQUEST_INTERVAL_SECONDS", "15"))
if not math.isfinite(INTERVAL) or INTERVAL < 0:
    raise ValueError("Request interval must be finite and nonnegative")
OUTPUT = ROOT / "eduassist_output"
OUTPUT.mkdir(exist_ok=True)
TRACKING_URI = "sqlite:///" + str((OUTPUT / "mlflow.db").resolve())
print("Existing .env loaded; credential value hidden.")
print("Outputs:", OUTPUT, "| Pause between requests:", INTERVAL, "seconds")

Existing .env loaded; credential value hidden.
Outputs: /Users/admin/Documents/ChatGPT/mlops-assignment2/Assignment_Submission/eduassist_output | Pause between requests: 15.0 seconds


## Questions, prompts and pricing
Use the five questions and three prompt templates below. Free-tier costs are zero; the paid-rate calculation is included as a separate comparison. The pricing source and date are recorded with the configuration. JSON output is requested through the prompt so that its validity can be measured.

In [3]:
import json
import math

PROMPTS = {
    'v1': 'Answer the following student question clearly.\nQuestion: {{question}}',
    'v2': 'You are an academic assistant.\nAnswer the following student question in a clear, concise, and accurate manner.\nUse simple language suitable for undergraduate students.\nQuestion: {{question}}',
    'v3': '''You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}''',
}

def validate(text):
    """Separate JSON syntax from the stronger EduAssist schema contract."""
    try:
        value = json.loads(text, parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))
    except (ValueError, TypeError):
        return False, False
    schema = (isinstance(value, dict) and set(value) == {'answer', 'confidence', 'key_points'}
              and isinstance(value['answer'], str) and bool(value['answer'].strip())
              and value['confidence'] in ('high', 'medium', 'low')
              and isinstance(value['key_points'], list) and bool(value['key_points'])
              and all(isinstance(x, str) and bool(x.strip()) for x in value['key_points']))
    return True, bool(schema)

def costs(input_tokens, output_tokens, cached_tokens, pricing):
    if not 0 <= cached_tokens <= input_tokens:
        raise ValueError('Invalid cached token count')
    a = ((input_tokens-cached_tokens)*pricing['input_per_million']
         + cached_tokens*pricing['cached_input_per_million'])/1_000_000
    b = output_tokens*pricing['output_per_million']/1_000_000
    return a, b, a+b

QUESTIONS = [{'id': 'Q1', 'question': 'What is overfitting in machine learning and how can it be reduced?'}, {'id': 'Q2', 'question': 'Explain the difference between supervised and unsupervised learning.'}, {'id': 'Q3', 'question': 'What is the purpose of cross-validation?'}, {'id': 'Q4', 'question': 'Explain precision and recall with an example.'}, {'id': 'Q5', 'question': 'How does gradient descent minimize a loss function?'}]
PRICING = {'model': 'gemini-3.5-flash-lite', 'provider': 'Google Gemini Developer API', 'tier': 'free', 'input_per_million': 0, 'cached_input_per_million': 0, 'output_per_million': 0, 'currency': 'USD', 'pricing_date': '2026-10-31', 'source': 'https://ai.google.dev/gemini-api/docs/pricing', 'api': 'Gemini Developer API REST v1beta generateContent', 'paid_equivalent': {'input_per_million': 0.3, 'cached_input_per_million': 0.03, 'output_per_million': 2.5}, 'note': 'Zero rates assume a free-tier project with available quota. Paid equivalent is a separate standard text-rate scenario, not an actual charge. No explicit caching, grounding, tools or paid fallback is used.'}
MODEL = PRICING["model"]
print("Model:", MODEL, "| API:", PRICING["api"], "| Currency:", PRICING["currency"])
print("Pricing date:", PRICING["pricing_date"], "| Source:", PRICING["source"])
display(pd.DataFrame(QUESTIONS))
for label, template in PROMPTS.items(): print(label + "\n" + template + "\n")

Model: gemini-3.5-flash-lite | API: Gemini Developer API REST v1beta generateContent | Currency: USD
Pricing date: 2026-10-31 | Source: https://ai.google.dev/gemini-api/docs/pricing


,id,question
0,Q1,What is overfitting in machine learning and ho...
1,Q2,Explain the difference between supervised and ...
2,Q3,What is the purpose of cross-validation?
3,Q4,Explain precision and recall with an example.
4,Q5,How does gradient descent minimize a loss func...


v1
Answer the following student question clearly.
Question: {{question}}

v2
You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.
Question: {{question}}

v3
You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}



## Helper functions
The functions below call Gemini, read its token counts and evaluate the responses. Request timing excludes the pause between calls. Errors are recorded in the results.

In [4]:
"""Gemini Developer API REST adapter (standard library, no SDK retries)."""
import json
import ssl
import certifi
import urllib.request
from urllib.parse import quote


def parse_response(data, requested_model):
    usage = data.get('usageMetadata')
    if not usage or 'promptTokenCount' not in usage:
        raise ValueError('Missing Gemini token usage')
    candidates = data.get('candidates') or []
    candidate = candidates[0] if candidates else {}
    text = ''.join(p.get('text', '') for p in candidate.get('content', {}).get('parts', [])
                   if not p.get('thought', False))
    thoughts = usage.get('thoughtsTokenCount', 0) or 0
    visible = usage.get('candidatesTokenCount', 0) or 0
    return dict(response=text, input_tokens=usage['promptTokenCount'],
                output_tokens=visible + thoughts, thinking_tokens=thoughts,
                cached_tokens=usage.get('cachedContentTokenCount', 0) or 0,
                total_tokens=usage.get('totalTokenCount', usage['promptTokenCount'] + visible + thoughts),
                model=data.get('modelVersion', requested_model),
                finish_reason=candidate.get('finishReason', 'NO_CANDIDATE'))


def generate(api_key, model, prompt):
    payload = {'contents': [{'role': 'user', 'parts': [{'text': prompt}]}],
               'generationConfig': {'temperature': 0, 'maxOutputTokens': 1024,
                                    'thinkingConfig': {'thinkingLevel': 'MINIMAL'}}}
    request = urllib.request.Request(
        'https://generativelanguage.googleapis.com/v1beta/models/'
        + quote(model, safe='') + ':generateContent',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json', 'x-goog-api-key': api_key}, method='POST')
    with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT, context=ssl.create_default_context(cafile=certifi.where())) as response:
        return parse_response(json.load(response), model)


In [5]:
import mlflow
from mlflow import genai
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)
mlflow.set_experiment("eduassist_llmops_assignment")
PROMPT_NAME = "eduassist_academic_qa"
BATCH = uuid.uuid4().hex[:12]
REGISTERED = {}
for label, template in PROMPTS.items():
    REGISTERED[label] = genai.register_prompt(
        name=PROMPT_NAME, template=template,
        commit_message={"v1":"v1 Basic: academic question baseline",
                        "v2":"v2 Structured: role, brevity and undergraduate audience",
                        "v3":"v3 JSON: require exact EduAssist schema"}[label])
    genai.set_prompt_version_tag(name=PROMPT_NAME,version=REGISTERED[label].version,
                                key="assignment_label",value=label)
ROWS = []
_LAST_END = None

def evaluate_version(label):
    global _LAST_END
    if any(row["prompt_version"] == label for row in ROWS):
        raise RuntimeError("This prompt was already evaluated in this kernel. Restart and Run All for a new complete batch.")
    for q in QUESTIONS:
        if _LAST_END is not None:
            time.sleep(max(0, INTERVAL - (time.perf_counter() - _LAST_END)))
        row = dict(question_id=q["id"], question=q["question"], prompt_version=label,
                   registry_version=int(REGISTERED[label].version), model=MODEL,
                   start_time=datetime.now(timezone.utc).isoformat(), status="error",
                   input_tokens=None,output_tokens=None,total_tokens=None,cached_tokens=None,
                   thinking_tokens=None,input_cost=None,output_cost=None,total_cost=None,
                   paid_equivalent_cost=None,response="",valid_json=0,schema_valid=0,
                   finish_reason="",http_status=None,error_type="")
        prompt = REGISTERED[label].format(question=q["question"])
        start = time.perf_counter()
        try:
            usage = generate(API_KEY, MODEL, prompt)
            a,b,c = costs(usage["input_tokens"],usage["output_tokens"],usage["cached_tokens"],PRICING)
            _,_,paid = costs(usage["input_tokens"],usage["output_tokens"],usage["cached_tokens"],PRICING["paid_equivalent"])
            valid,schema = validate(usage["response"])
            row.update(usage,status="success" if usage["finish_reason"]=="STOP" else "incomplete",
                       http_status=200,input_cost=a,output_cost=b,total_cost=c,
                       paid_equivalent_cost=paid,valid_json=int(valid),schema_valid=int(schema))
        except Exception as exc:
            row.update(error_type=type(exc).__name__,http_status=getattr(exc,"code",None))
        row.update(end_time=datetime.now(timezone.utc).isoformat(),latency_seconds=time.perf_counter()-start)
        _LAST_END = time.perf_counter()
        ROWS.append(row)
        pd.DataFrame(ROWS).to_csv(OUTPUT / "all_requests.csv",index=False)
        print(label,q["id"],row["status"],"HTTP",row["http_status"],flush=True)
        if row["http_status"] in (400,401,403,404,429) or row["error_type"]=="URLError":
            raise RuntimeError(f"Gemini request failed (HTTP {row['http_status']}, {row['error_type']}); partial evidence is saved. Check account access, model availability, quota or connectivity.")
    return pd.DataFrame([r for r in ROWS if r["prompt_version"]==label]).sort_values("question_id")
print("Three exact prompts registered. No inference requests made yet.")

2026/11/04 23:55:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/04 23:55:21 INFO mlflow.store.db.utils: Updating database tables
2026-11-04 23:55:21 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2026-11-04 23:55:21 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2026-11-04 23:55:21 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2026-11-04 23:55:21 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2026/11/04 23:55:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/04 23:55:21 INFO mlflow.store.db.utils: Updating database tables
2026-11-04 23:55:21 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2026-11-04 23:55:21 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Three exact prompts registered. No inference requests made yet.


## Task 1: API calls and latency
Run v1 on five questions. Record the request details and calculate the mean, median, minimum, maximum and sample standard deviation of latency. Identify the slowest request and discuss possible reasons.

In [6]:
baseline = evaluate_version("v1")
baseline.to_csv(OUTPUT / "latency_tokens_cost.csv",index=False)
display(baseline[["question_id","prompt_version","model","start_time","end_time","latency_seconds","status"]])
baseline_stats = baseline.latency_seconds.agg(["mean","median","min","max","std"]).rename_axis("statistic").to_frame("seconds")
baseline_stats.to_csv(OUTPUT / "task1_latency_summary.csv")
display(baseline_stats)
slow = baseline.loc[baseline.latency_seconds.idxmax()]
TASK1_INTERPRETATION = (f"The slowest v1 baseline request was {slow.question_id} at {slow.latency_seconds:.3f} seconds, "
    f"with {slow.output_tokens} output tokens and status {slow.status}. Longer generation, network variation, "
    "connection establishment and provider queueing are possible causes; this experiment does not prove which caused the difference.")
print(TASK1_INTERPRETATION)
(OUTPUT / "task1_interpretation.txt").write_text(TASK1_INTERPRETATION)

v1 Q1 success HTTP 200
v1 Q2 success HTTP 200
v1 Q3 success HTTP 200
v1 Q4 success HTTP 200
v1 Q5 success HTTP 200


,question_id,prompt_version,model,start_time,end_time,latency_seconds,status
0,Q1,v1,gemini-3.5-flash-lite,2026-11-04T18:25:22.100808+00:00,2026-11-04T18:25:51.876312+00:00,29.775392,success
1,Q2,v1,gemini-3.5-flash-lite,2026-11-04T18:26:06.878174+00:00,2026-11-04T18:26:17.753199+00:00,10.874939,success
2,Q3,v1,gemini-3.5-flash-lite,2026-11-04T18:26:32.756593+00:00,2026-11-04T18:26:52.061113+00:00,19.304459,success
3,Q4,v1,gemini-3.5-flash-lite,2026-11-04T18:27:07.066213+00:00,2026-11-04T18:27:24.622990+00:00,17.556680,success
4,Q5,v1,gemini-3.5-flash-lite,2026-11-04T18:27:39.628090+00:00,2026-11-04T18:27:51.247439+00:00,11.619254,success


,seconds
statistic,
mean,17.826145
median,17.556680
min,10.874939
max,29.775392
std,7.613923


The slowest v1 baseline request was Q1 at 29.775 seconds, with 628 output tokens and status success. Longer generation, network variation, connection establishment and provider queueing are possible causes; this experiment does not prove which caused the difference.


266

## Task 2: Token usage and cost
Calculate input and output costs from the reported token counts and rates per million tokens. The monthly estimate is the average request cost multiplied by 10,000. This is a cost projection, not a guarantee of free-tier quota.

In [7]:
display(baseline[["question_id","input_tokens","output_tokens","total_tokens","cached_tokens","input_cost","output_cost","total_cost","paid_equivalent_cost"]])
BASELINE_MONTHLY = baseline.total_cost.mean() * 10000
PAID_BASELINE_MONTHLY = baseline.paid_equivalent_cost.mean() * 10000
print(f"Baseline monthly projection at free-tier rates: USD {BASELINE_MONTHLY:.6f}")
print(f"Separate paid-equivalent monthly scenario: USD {PAID_BASELINE_MONTHLY:.6f}")
print("Optimization: shorten verbose responses while preserving academic accuracy and required fields; re-evaluate quality before lowering the output limit.")

,question_id,input_tokens,output_tokens,total_tokens,cached_tokens,input_cost,output_cost,total_cost,paid_equivalent_cost
0,Q1,24,628,652,0,0.0,0.0,0.0,0.001577
1,Q2,20,563,583,0,0.0,0.0,0.0,0.001413
2,Q3,20,391,411,0,0.0,0.0,0.0,0.000983
3,Q4,19,723,742,0,0.0,0.0,0.0,0.001813
4,Q5,20,436,456,0,0.0,0.0,0.0,0.001096


Baseline monthly projection at free-tier rates: USD 0.000000
Separate paid-equivalent monthly scenario: USD 13.766800
Optimization: shorten verbose responses while preserving academic accuracy and required fields; re-evaluate quality before lowering the output limit.


## Task 3: Prompt comparison and MLflow
Evaluate v2 and v3 on the same five questions and compare them with v1. Log the five required metrics, register the prompts and assign the production alias to the selected version.

`valid_json` is the number of valid JSON responses divided by five. A separate schema check verifies the required fields and types. JSON validity alone does not establish factual accuracy.

In [8]:
structured = evaluate_version("v2")
json_answers = evaluate_version("v3")
requests = pd.concat([baseline,structured,json_answers],ignore_index=True)
assert len(requests)==15
assert all(set(g.question_id)==set(baseline.question_id) for _,g in requests.groupby("prompt_version"))
requests.to_csv(OUTPUT / "all_requests.csv",index=False)
comparisons = []
for label,group in requests.groupby("prompt_version",sort=True):
    ok = group[group.status=="success"]
    metrics = dict(avg_latency_seconds=float(group.latency_seconds.mean()),
        avg_input_tokens=float(ok.input_tokens.mean()) if len(ok) else 0.,
        avg_output_tokens=float(ok.output_tokens.mean()) if len(ok) else 0.,
        avg_cost_per_request=float(ok.total_cost.mean()) if len(ok) else 0.,
        avg_paid_equivalent_cost=float(ok.paid_equivalent_cost.mean()) if len(ok) else 0.,
        valid_json=float(group.valid_json.sum()/5),schema_valid=float(group.schema_valid.sum()/5),
        success_rate=float(len(ok)/5))
    with mlflow.start_run(run_name=f"{BATCH}_{label}") as active:
        mlflow.log_params(dict(prompt_label=label,prompt_name=PROMPT_NAME,registry_version=int(REGISTERED[label].version),
            model=MODEL,api=PRICING["api"],pricing_date=PRICING["pricing_date"],currency="USD",pricing_tier="free",
            batch=BATCH,temperature=0,max_output_tokens=1024,thinking_level="MINIMAL",request_interval_seconds=INTERVAL))
        mlflow.log_metrics(metrics)
        mlflow.log_text(PROMPTS[label],"prompt.txt")
        mlflow.log_dict(PRICING,"pricing.json")
        mlflow.log_dict(QUESTIONS,"questions.json")
        mlflow.log_table(group.to_dict("list"),"requests.json")
        comparisons.append(dict(prompt_version=label,registry_version=int(REGISTERED[label].version),run_id=active.info.run_id,
            **metrics,projected_monthly_cost=metrics["avg_cost_per_request"]*10000,
            paid_equivalent_monthly_cost=metrics["avg_paid_equivalent_cost"]*10000))
comparison = pd.DataFrame(comparisons)
eligible = comparison[(comparison.success_rate==1)&(comparison.schema_valid==1)]
winner = None if eligible.empty else eligible.sort_values(["avg_cost_per_request","avg_latency_seconds"]).iloc[0].prompt_version
comparison["recommended"] = comparison.prompt_version==winner
comparison.to_csv(OUTPUT / "prompt_evaluation_results.csv",index=False)
if winner:
    genai.set_prompt_alias(name=PROMPT_NAME,alias="production",version=int(REGISTERED[winner].version))
    promoted = genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    assert int(promoted.version)==int(REGISTERED[winner].version)
requests.groupby("prompt_version").latency_seconds.agg(["mean","median","min","max","std"]).to_csv(OUTPUT / "latency_summary.csv")
diff="\n".join("".join(difflib.unified_diff(PROMPTS[a].splitlines(True),PROMPTS[b].splitlines(True),fromfile=a,tofile=b)) for a,b in [("v1","v2"),("v2","v3")])
(OUTPUT / "prompt_diff.txt").write_text(diff)
manifest = dict(mode="live",batch=BATCH,tracking_uri=TRACKING_URI,prompt_name=PROMPT_NAME,winner=winner,
    registry_versions={k:int(v.version) for k,v in REGISTERED.items()},pricing=PRICING,
    complete=bool((requests.status=="success").all()),total_requests=len(requests),task1_prompt="v1",task1_requests=5)
(OUTPUT / "manifest.json").write_text(json.dumps(manifest,indent=2))
display(comparison[["prompt_version","avg_latency_seconds","avg_input_tokens","avg_output_tokens","avg_cost_per_request","valid_json","schema_valid","recommended"]])
display(comparison[["prompt_version","projected_monthly_cost","paid_equivalent_monthly_cost"]])
print("v3 valid_json:", int(json_answers.valid_json.sum()), "/", len(json_answers), "=", json_answers.valid_json.mean())
print("Production candidate:",winner or "No eligible version; no new alias assigned")
print(diff)
pd.set_option("display.max_colwidth",None)
display(requests[["question_id","prompt_version","response","valid_json","schema_valid"]])

v2 Q1 success HTTP 200
v2 Q2 success HTTP 200
v2 Q3 success HTTP 200
v2 Q4 success HTTP 200
v2 Q5 success HTTP 200
v3 Q1 success HTTP 200
v3 Q2 success HTTP 200
v3 Q3 success HTTP 200
v3 Q4 success HTTP 200
v3 Q5 success HTTP 200


,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request,valid_json,schema_valid,recommended
0,v1,17.826145,20.6,548.2,0.0,0.0,0.0,False
1,v2,18.699385,44.6,392.8,0.0,0.0,0.0,False
2,v3,12.440708,85.6,141.2,0.0,1.0,1.0,True


,prompt_version,projected_monthly_cost,paid_equivalent_monthly_cost
0,v1,0.0,13.7668
1,v2,0.0,9.9538
2,v3,0.0,3.7868


v3 valid_json: 5 / 5 = 1.0
Production candidate: v3
--- v1
+++ v2
@@ -1,2 +1,4 @@
-Answer the following student question clearly.
+You are an academic assistant.
+Answer the following student question in a clear, concise, and accurate manner.
+Use simple language suitable for undergraduate students.
 Question: {{question}}
--- v2
+++ v3
@@ -1,4 +1,10 @@
 You are an academic assistant.
-Answer the following student question in a clear, concise, and accurate manner.
-Use simple language suitable for undergraduate students.
+Answer the student question and return ONLY valid JSON.
+Do not include markdown, explanations outside JSON, or code fences.
+Use this exact JSON schema:
+{
+  "answer": "string",
+  "confidence": "high | medium | low",
+  "key_points": ["string"]
+}
 Question: {{question}}


,question_id,prompt_version,response,valid_json,schema_valid
0,Q1,v1,"Here is a clear and simple explanation of **overfitting** in machine learning, along with ways to prevent it.\n\n---\n\n### **What is Overfitting?**\n\n**Overfitting** happens when a machine learning model learns the training data *too well*. \n\nInstead of learning the general underlying patterns, the model memorizes the specific details—including the **noise** (random fluctuations) and **outliers** (unusual data points) in the training set. \n\n* **The Analogy:** Imagine a student who memorizes the exact answers to last year’s math test instead of learning the formulas. When given a brand-new test with slightly different numbers, the student fails because they don't actually understand the concepts.\n* **The Result:** An overfitted model has **high accuracy on training data**, but **poor performance on new, unseen data (testing data)**. It lacks ""generalization.""\n\n---\n\n### **How to Reduce Overfitting**\n\nTo prevent a model from overfitting, you want to discourage it from becoming too complex. Here are the most common techniques:\n\n#### **1. Use More Training Data**\nThe easiest way to prevent overfitting is to feed the model more data. The more diverse examples the model sees, the harder it is for it to memorize the data, forcing it to learn generalized patterns instead.\n\n#### **2. Simplify the Model (Pruning / Fewer Parameters)**\nIf a model is too complex (e.g., a deep neural network with millions of parameters or a high-degree polynomial), it can easily memorize noise. Reducing the complexity of the model helps it focus only on the most important trends.\n\n#### **3. Regularization (L1 and L2)**\nRegularization adds a ""penalty"" to the model's loss function for having overly large or complex coefficients. \n* **L2 Regularization (Ridge):** Shrinks the coefficients of less important features close to zero.\n* **L1 Regularization (Lasso):** Can drive the coefficients of useless features completely to zero, effectively removing them.\n\n#### **4. Cross-Validation**\nInstead of using a single train/test split, use **K-Fold Cross-Validation**. This technique splits your data into multiple parts, trains the model on different combinations of the data, and validates it on the rest. This ensures your model's performance is consistent and not just a result of a lucky data split.\n\n#### **5. Dropout (for Neural Networks)**\nDuring training, ""dropout"" randomly turns off a percentage of neurons in a neural network at each step. This prevents neurons from co-adapting too much and forces the network to learn robust features through different pathways.\n\n#### **6. Early Stopping**\nWhen training an iterative model (like a Neural Network or Gradient Boosting), the model's performance on validation data improves up to a point, and then starts getting worse as it begins to overfit. **Early stopping** monitors the validation loss and halts the training process the moment it starts to increase.",0,0
1,Q2,v1,"Here is a clear and simple explanation of the difference between supervised and unsupervised learning:\n\n### **1. Supervised Learning**\n* **The Concept:** The model is trained using **labeled data**. This means that for every input data point, the correct output (or ""answer"") is already provided to the algorithm. \n* **The Analogy:** Think of a student learning with a teacher. The teacher gives the student flashcards with pictures of animals *along with* the correct labels (""This is a cat,"" ""This is a dog""). The student learns by checking their guesses against the correct answers.\n* **Common Tasks:** \n * *Classification* (predicting a category/label, e.g., Is this email ""Spam"" or ""Not Spam""?).\n * *Regression* (predicting a continuous number, e.g., What will the price of a house be?).\n* **Common Algorithms:** Linear Regression, Logistic Regression, Support Vector Machines (SVM), Random Forests.\n\n---\n\n### **2. Unsupervised Learning**\n*